### Testing CHIEF Features

This code base will do the following things:

1. Test the extraction of patch features of the image using CTransPath model.
2. Use multiple patches of the images extracted using tiling (via Histolab repo)
3. Extract the patch features of each patch and store in a .pt file
4. Pass the features of all the patches to the CHIEF model to get the attention of each patch
5. Draw a heatmap depicting the attention of each patch

In [1]:
import sys
import os

Adding the directory CHIEF to use the modules

In [2]:
chief_path = os.path.join(os.getcwd(), 'CHIEF')
sys.path.extend([chief_path,])

In [3]:
import os
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from models.ctran import ctranspath
import numpy as np
import matplotlib.pyplot as plt

### Extracting the Patch Features

Taking some default settings from CTransPath

In [55]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
trnsfrms_val = transforms.Compose(
    [
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean = mean, std = std)
    ]
)

Loading the weights of the model `CTransPath`

In [56]:
model = ctranspath()
model.head = nn.Identity()
td = torch.load(r'./CHIEF/model_weight/CHIEF_CTransPath.pth')
model.load_state_dict(td['model'], strict=True)
model.eval()

SwinTransformer(
  (patch_embed): ConvStem(
    (proj): Sequential(
      (0): Conv2d(3, 12, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(12, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(12, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (4): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
      (6): Conv2d(24, 96, kernel_size=(1, 1), stride=(1, 1))
    )
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (layers): Sequential(
    (0): BasicLayer(
      dim=96, input_resolution=(56, 56), depth=2
      (blocks): ModuleList(
        (0): SwinTransformerBlock(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): WindowAttention(
            (qkv): Linear(in_features=96, out_features=288, bias=True

#### Reading the patches of the WSI

In [49]:
tile_folder = './ovarian/processed/grid'
tile_names = os.listdir(tile_folder)
tile_tensors = []

for tile in tile_names:
    tile_path = os.path.join(tile_folder, tile)
    tile_orig = Image.open(tile_path).convert('RGB')
    tile_transform = trnsfrms_val(tile_orig).unsqueeze(dim=0)
    tile_tensors.append(tile_transform)

tile_tensors = torch.cat(tile_tensors, dim=0)

In [53]:
print('Tile Tensors:', tile_tensors.shape)

Tile Tensors: torch.Size([8594, 3, 224, 224])


Getting the patch embeddings from CTransPath

In [57]:
with torch.no_grad():
    patch_feature_emb = model(tile_tensors) # Extracted features (torch.Tensor) with shape [1,768]
    print(patch_feature_emb.size())

torch.Size([8594, 768])


Saving the generated embeddings to be used by WSI Feature of CHIEF

In [58]:
torch.save(patch_feature_emb, './ovarian/processed/patch_feature_emb_grid.pt')

### Extracting the WSI Feature and attention score associated with each patch

In [59]:
from models.CHIEF import CHIEF

In [60]:
model = CHIEF(size_arg="small", dropout=True, n_classes=2)

td = torch.load(r'./CHIEF/model_weight/CHIEF_pretraining.pth')
model.load_state_dict(td, strict=True)
model.eval()

[768, 512, 256]


CHIEF(
  (attention_net): Sequential(
    (0): Linear(in_features=768, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
    (3): Attn_Net_Gated(
      (attention_a): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Tanh()
        (2): Dropout(p=0.25, inplace=False)
      )
      (attention_b): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Sigmoid()
        (2): Dropout(p=0.25, inplace=False)
      )
      (attention_c): Linear(in_features=256, out_features=1, bias=True)
    )
  )
  (classifiers): Linear(in_features=512, out_features=2, bias=True)
  (instance_classifiers): ModuleList(
    (0-1): 2 x Linear(in_features=512, out_features=2, bias=True)
  )
  (instance_loss_fn): CrossEntropyLoss()
  (att_head): Att_Head(
    (fc1): Linear(in_features=512, out_features=256, bias=True)
    (relu): ReLU()
    (fc2): Linear(in_features=256, out_features=1, bias=True)
    (s

Loading the feature vector

In [61]:
feature_path = r'./ovarian/processed/patch_feature_emb_grid.pt'

patch_feature_emb = torch.load(feature_path, map_location=torch.device('cpu'))

In [64]:
anatomical=13
with torch.no_grad():
    x,tmp_z = patch_feature_emb,anatomical
    result = model(x, torch.tensor([tmp_z]))
    wsi_feature_emb = result['WSI_feature']  ###[1,768]
    print(wsi_feature_emb.size())

torch.Size([1, 768])


In [65]:
print("Raw Attention: ", result['attention_raw'])
print("Raw Attention (Softmax): ", torch.nn.functional.softmax(result['attention_raw'], dim=1))

Raw Attention:  tensor([[-0.1173,  1.5326,  0.1642,  ...,  0.5233,  0.2439,  0.7065]])
Raw Attention (Softmax):  tensor([[2.5758e-05, 1.3409e-04, 3.4132e-05,  ..., 4.8877e-05, 3.6962e-05,
         5.8700e-05]])
